# 01 - Clean the data (Phase 2 in Fabric)
**Before you run:** create a Lakehouse, attach it as this notebook's default Lakehouse, and upload into its **Files** area:
- the `src/` folder (all `.py` files)
- `data/raw/enquiries_raw.csv`
- `data/ground_truth/` (both CSV files)

This notebook runs the same `data_quality.py` as the local edition, then saves the results as Lakehouse (Delta) tables.

In [ ]:
import sys
ROOT = "/lakehouse/default/Files"          # the attached Lakehouse's Files area
sys.path.insert(0, f"{ROOT}/src")          # so we can import the same .py files as the local edition
%pip install -q anthropic python-dotenv pydantic faker

In [ ]:
# Optional: regenerate the synthetic data instead of uploading it
# import generate_data; generate_data.main()

In [ ]:
import data_quality
data_quality.main()     # reads Files/data/raw, writes Files/data/clean (no API key = demo mode)

In [ ]:
print(open(f"{ROOT}/data/clean/quality_report.md").read()[:2500])

## Save the cleaned data as Delta tables (these appear in the SQL analytics endpoint)

In [ ]:
def csv_to_table(path, table):
    df = (spark.read.option("header", True).option("inferSchema", True)
          .option("multiLine", True).option("escape", '"').csv(path))
    df.write.mode("overwrite").format("delta").saveAsTable(table)
    print(table, df.count(), "rows")

csv_to_table("Files/data/clean/enquiries_clean.csv", "enquiries")
csv_to_table("Files/data/clean/data_quality_log.csv", "data_quality_log")
csv_to_table("Files/data/clean/quality_scores.csv", "quality_scores")